### Libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from IPython.display import Audio

: 

#### Nonnegative Matrix Factorization (NMF)

In [ ]:
# Generate a sample non-negative matrix V (e.g., a 4x6 matrix)
np.random.seed(0)
V = np.random.rand(4, 6)

# Set the rank for the factorization
r = 2  # Number of basis components (features)

# Initialize W and H with random values
m, n = V.shape
W = np.random.rand(m, r)
H = np.random.rand(r, n)

# Define the number of iterations and a small threshold to stop the algorithm
max_iterations = 1000
tolerance = 1e-5

# Multiplicative Update Rules for NMF
for iteration in range(max_iterations):
    # Update H (keeping W fixed)
    H *= (W.T @ V) / (W.T @ W @ H + 1e-10)  # Adding 1e-10 to avoid division by zero

    # Update W (keeping H fixed)
    W *= (V @ H.T) / (W @ H @ H.T + 1e-10)  # Adding 1e-10 to avoid division by zero

    # Calculate the reconstruction error using Frobenius norm
    reconstruction_error = np.linalg.norm(V - W @ H, 'fro')

    # Check if the error is below the tolerance level
    if reconstruction_error < tolerance:
        print(f'Converged in {iteration} iterations')
        break

# Print the results
print("Original matrix V:")
print(V)
print("\nFactorized matrix W:")
print(W)
print("\nFactorized matrix H:")
print(H)
print("\nReconstructed matrix (W @ H):")
print(W @ H)
print("\nReconstruction error:", reconstruction_error)

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original matrix V
axes[0].imshow(V, aspect='auto', cmap='viridis')
axes[0].set_title('Original Matrix V')
axes[0].set_xlabel('Columns')
axes[0].set_ylabel('Rows')

# Factorized matrix W
axes[1].imshow(W, aspect='auto', cmap='viridis')
axes[1].set_title('Factorized Matrix W')
axes[1].set_xlabel('Components')
axes[1].set_ylabel('Rows')

# Factorized matrix H
axes[2].imshow(H, aspect='auto', cmap='viridis')
axes[2].set_title('Factorized Matrix H')
axes[2].set_xlabel('Columns')
axes[2].set_ylabel('Components')

# Reconstructed matrix (W @ H)
reconstructed = W @ H
axes[3].imshow(reconstructed, aspect='auto', cmap='viridis')
axes[3].set_title('Reconstructed Matrix (W @ H)')
axes[3].set_xlabel('Columns')
axes[3].set_ylabel('Rows')

plt.tight_layout()
plt.show()


### Introducing FastMNMF

In [ ]:
# Load multichannel audio
file_path = '/content/multichannel.wav'
y, sr = sf.read(file_path)

# Check if the audio has multiple channels
if y.ndim < 2:
    raise ValueError("The provided audio file does not have multiple channels.")

num_channels = y.shape[1]  # Number of channels
V = []  # List to store spectrograms for each channel

# Convert each channel to a spectrogram
for c in range(num_channels):
    channel_audio = y[:, c]
    spectrogram = np.abs(librosa.stft(channel_audio, n_fft=1024, hop_length=512))**2
    V.append(spectrogram)

# Set rank for the factorization
r = 5  # Higher rank may help separate distinct sources
m, n = V[0].shape
W = np.random.rand(m, r)
H = np.random.rand(r, n)
Sigma = [np.eye(r) for _ in range(num_channels)]

# Define number of iterations and tolerance
max_iterations = 1000
tolerance = 1e-5

# MNMF Iterative Updates
for iteration in range(max_iterations):
    numerator_H = sum([W.T @ V[c] for c in range(num_channels)])
    denominator_H = sum([W.T @ W @ H for c in range(num_channels)]) + 1e-10
    H *= numerator_H / denominator_H

    numerator_W = sum([V[c] @ H.T for c in range(num_channels)])
    denominator_W = sum([W @ H @ H.T for c in range(num_channels)]) + 1e-10
    W *= numerator_W / denominator_W

    for c in range(num_channels):
        error = V[c] - W @ H
        Sigma[c] = np.diag(np.sum(error ** 2, axis=0)[:r]) / (np.sum(H ** 2, axis=1) + 1e-10)

    reconstruction_error = sum([np.linalg.norm(V[c] - W @ H, 'fro') for c in range(num_channels)])

    if reconstruction_error < tolerance:
        print(f'Converged in {iteration} iterations')
        break

# Visualize Original and Reconstructed Matrices for each Channel
fig, axes = plt.subplots(num_channels, 2, figsize=(10, 5 * num_channels))

for c in range(num_channels):
    axes[c, 0].imshow(librosa.amplitude_to_db(V[c], ref=np.max), aspect='auto', cmap='viridis')
    axes[c, 0].set_title(f'Original Matrix V (Channel {c+1})')
    axes[c, 0].set_xlabel('Columns')
    axes[c, 0].set_ylabel('Rows')

    reconstructed = W @ H
    axes[c, 1].imshow(librosa.amplitude_to_db(reconstructed, ref=np.max), aspect='auto', cmap='viridis')
    axes[c, 1].set_title(f'Reconstructed Matrix (Channel {c+1})')
    axes[c, 1].set_xlabel('Columns')
    axes[c, 1].set_ylabel('Rows')

plt.tight_layout()
plt.savefig('mnmf_result.png', format='png', dpi=300)
plt.show()

# Convert reconstructed spectrograms back to audio and save
for c in range(num_channels):
    reconstructed_audio = librosa.istft(np.sqrt(reconstructed), hop_length=512)
    output_path = f'reconstructed_channel_{c+1}.wav'
    sf.write(output_path, reconstructed_audio, sr)
    print(f"Saved reconstructed audio for channel {c+1} to {output_path}")

In [ ]:
from IPython.display import Audio

# Play reconstructed audio for each channel
for c in range(1, 7):  # Adjust range as per the number of channels
    file_path = f'reconstructed_channel_{c}.wav'
    print(f"Playing audio for reconstructed_channel_{c}.wav")
    display(Audio(filename=file_path))

### Audio Playback for comparison 

In [ ]:
# Load the original multichannel audio file
file_path = '/content/multichannel.wav'
y, sr = sf.read(file_path)  # Read the audio file and sampling rate

# Play each original channel
print("Original Channels:")
for c in range(y.shape[1]):  # Assuming y.shape[1] is the number of channels
    print(f"Playing original audio for channel {c+1}")
    display(Audio(data=y[:, c], rate=sr))

# Play each reconstructed channel
print("\nReconstructed Channels:")
for c in range(1, 7):  # Adjust range as per the number of saved reconstructed channels
    reconstructed_file_path = f'reconstructed_channel_{c}.wav'
    print(f"Playing reconstructed audio for channel {c}")
    display(Audio(filename=reconstructed_file_path))

: 